# 输出解析器

In [1]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_models.moonshot import MoonshotChat
from dotenv import load_dotenv
import os

In [5]:
load_dotenv()

llm = MoonshotChat(
    model=os.getenv('MOONSHOT_MODEL'), 
    verbose=True,
    temperature=0.7, 
    max_tokens=1024, 
    api_key=os.getenv('MOONSHOT_API_KEY')
)

## StrOutputParser

In [15]:
chat_template = ChatPromptTemplate.from_template(
    """
    你是一个耐心的大姐姐，要为我回答各种问题。
    我想问：{question}
    """
)

In [ ]:
parser = StrOutputParser()

In [16]:
chain = chat_template | llm | parser

In [17]:
res = chain.invoke("中国和美国的高中生有什么区别？")

In [18]:
print(res)

当然可以，小同学～这个问题很有意思，也很现实。中美高中生的差异，其实不只是“作业多不多”“考试难不难”那么简单，而是像两棵不同的树，根扎在不同的土壤里，长出来的枝叶自然也不一样。姐姐给你慢慢说：

---

### 1. **时间怎么过？**
- **中国高中生**：  
  时间像被“课表”切成一块块豆腐，早7晚10是常态。白天在教室“坐牢”，晚上在作业里“渡劫”。周末？很多学校是“大小周”——小周只放半天，大周放一天半。  
  最常说的话：“这题我刷过，但考试时还是错了。”

- **美国高中生**：  
  时间像“自助餐”。早上8点上课，下午3点放学，但之后是社团、体育、乐队、义工、打工……有人为了凑满“简历”，一天跑五个活动。  
  最常说的话：“我下周有橄榄球比赛，还有AP生物实验，可能没时间睡觉。”

---

### 2. **成绩怎么算？**
- **中国高中生**：  
  一考定乾坤。期中、期末、月考、联考……分数是“硬通货”，年级大榜贴走廊，谁上谁下，一目了然。  
  老师口头禅：“多考一分，干掉千人。”

- **美国高中生**：  
  成绩像“拼拼图”。平时作业占10%，小测占20%，项目占30%，期末考试只占15%。而且老师打分主观性强，论文写得好，老师一高兴就给你A。  
  老师口头禅：“I’m not grading you, I’m assessing your growth.”（“我不是在打分，我是在评估你的成长。”）

---

### 3. **同学关系？**
- **中国高中生**：  
  同班同学像“战友”，三年不换教室，一起抄笔记、一起吐槽老师、一起偷偷点外卖。但竞争也赤裸裸，有人把参考书藏起来，有人“卷”到凌晨三点。  
  暗号：“你昨晚几点睡的？”（其实想探口风）

- **美国高中生**：  
  同学像“流动摊位”。每节课换教室，认识的人散在不同“梯队”：有AP班的天才，也有普通班的“派对王”。竞争藏在水面下，大家嘴上都说“I don’t care”，但背地里也在拼GPA。  
  暗号：“Which college are you applying to?”（其实是“你SAT考了几分？”的礼貌版）

---

### 4. **老师和家长？**
- **中国老师**：  
  像“班主任+教官+

定义 Pydantic 数据模型，用于格式化大模型输出结果

## 严格的LLM输出格式限制

### 不采用LCEL的写法（以JsonOutputParser举例）

In [2]:
from pydantic import BaseModel, Field

In [3]:
class RecipeResult(BaseModel):
    dish_name: str = Field(description="菜品名称")
    cook_time: int = Field(description="烹饪时间(分钟)")
    ingredients: list[str] = Field(description="食材列表")

In [34]:
json_parser = JsonOutputParser(pydantic_object=RecipeResult)
format_instruction = json_parser.get_format_instructions()

template = ChatPromptTemplate.from_template(template="请提取菜谱信息。\n{format_instructions}\n用户输入：{query}")
query = "我最喜欢吃宫保鸡丁，这道菜需要鸡胸肉、萝卜丁、黄瓜丁、花生米和各种调料，妈妈每次做饭都要炒20分钟呢。"

prompt = template.format(format_instructions=format_instruction, query=query)
print(prompt)

Human: 请提取菜谱信息。
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"dish_name": {"description": "菜品名称", "title": "Dish Name", "type": "string"}, "cook_time": {"description": "烹饪时间(分钟)", "title": "Cook Time", "type": "integer"}, "ingredients": {"description": "食材列表", "items": {"type": "string"}, "title": "Ingredients", "type": "array"}}, "required": ["dish_name", "cook_time", "ingredients"]}
```
用户输入：我最喜欢吃宫保鸡丁，这道菜需要鸡胸肉、萝卜丁、黄瓜丁、花生米和各种调料，妈妈每次做饭都要炒20分钟呢。


In [37]:
raw_ai_message = llm.invoke(prompt)
print(raw_ai_message.content)

{"dish_name": "宫保鸡丁", "cook_time": 20, "ingredients": ["鸡胸肉", "萝卜丁", "黄瓜丁", "花生米", "各种调料"]}


In [44]:
json_result = json_parser.invoke(raw_ai_message)
print("\n--- 解析器最终转换出来的 Python 对象（事后） ---")
print(type(json_result))  # 类型是 <class '__main__.RecipeResult'>
print(f"菜名: {json_result['dish_name']}")
print(f"时间: {json_result['cook_time']} 分钟")
print(f"材料：{json_result['ingredients']}")


--- 解析器最终转换出来的 Python 对象（事后） ---
<class 'dict'>
菜名: 宫保鸡丁
时间: 20 分钟
材料：['鸡胸肉', '萝卜丁', '黄瓜丁', '花生米', '各种调料']


### 采用LCEL写法（以PydanticOutputParser举例）

In [4]:
class RecipeResult(BaseModel):
    dish_name: str = Field(description="菜品名称")
    cook_time: int = Field(description="烹饪时间(分钟)")
    ingredients: list[str] = Field(description="食材列表")

In [5]:
pydantic_parser = PydanticOutputParser(pydantic_object=RecipeResult)

In [6]:
template = ChatPromptTemplate.from_template("请提取菜谱信息。\n{format_instructions}\n用户输入：{query}")
prompt = template.partial(format_instructions=pydantic_parser.get_format_instructions())

In [65]:
chain = prompt | llm | pydantic_parser

In [66]:
parser_res = chain.invoke("我最喜欢吃宫保鸡丁，这道菜需要鸡胸肉、萝卜丁、黄瓜丁、花生米和各种调料，妈妈每次做饭都要炒20分钟呢。")

In [71]:
print("\n--- 解析器最终转换出来的 Python 对象（事后） ---")
print(type(parser_res))  # 类型是 <class '__main__.RecipeResult'>
print(f"菜名: {parser_res.dish_name}")
print(f"时间: {parser_res.cook_time} 分钟")
print(f"材料：{parser_res.ingredients}")


--- 解析器最终转换出来的 Python 对象（事后） ---
<class '__main__.RecipeResult'>
菜名: 宫保鸡丁
时间: 20 分钟
材料：['鸡胸肉', '萝卜丁', '黄瓜丁', '花生米', '各种调料']
